# Import packages

In [40]:
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

from xgboost import XGBClassifier

# Load Data

In [41]:
# Set base path and data path and read event_info and feature_description
base_path = "wind_turbine_fault_detection\Wind_Farm_A"
data_path = os.path.join(base_path, "datasets")
event_info = pd.read_csv(os.path.join(base_path, "event_info.csv"), sep=';')
feature_description = pd.read_csv(os.path.join(base_path, "feature_description.csv"), sep=';')

<>:2: SyntaxWarning: invalid escape sequence '\W'
<>:2: SyntaxWarning: invalid escape sequence '\W'
C:\Users\usuario\AppData\Local\Temp\ipykernel_17556\3185466120.py:2: SyntaxWarning: invalid escape sequence '\W'
  base_path = "wind_turbine_fault_detection\Wind_Farm_A"


In [42]:
event_info.head()

,asset,event_id,event_label,event_start,event_start_id,event_end,event_end_id,event_description
0,11,68,anomaly,2023-07-28 13:20:00,52063,2023-08-11 13:10:00,54076,Transformer failure
1,21,22,anomaly,2023-08-12 09:50:00,51888,2023-08-19 10:00:00,52892,Hydraulic group
2,21,72,anomaly,2023-10-10 08:40:00,52497,2023-10-17 08:40:00,53505,Gearbox failure
3,0,73,anomaly,2023-06-10 11:40:00,52745,2023-06-17 11:40:00,53753,Hydraulic group
4,0,0,anomaly,2023-08-06 06:10:00,52436,2023-08-20 06:10:00,54447,Generator bearing failure


In [ ]:
event_info.head()

   asset  event_id event_label          event_start  event_start_id  \
0     11        68     anomaly  2023-07-28 13:20:00           52063   
1     21        22     anomaly  2023-08-12 09:50:00           51888   
2     21        72     anomaly  2023-10-10 08:40:00           52497   
3      0        73     anomaly  2023-06-10 11:40:00           52745   
4      0         0     anomaly  2023-08-06 06:10:00           52436   

             event_end  event_end_id          event_description  
0  2023-08-11 13:10:00         54076        Transformer failure  
1  2023-08-19 10:00:00         52892            Hydraulic group  
2  2023-10-17 08:40:00         53505            Gearbox failure  
3  2023-06-17 11:40:00         53753            Hydraulic group  
4  2023-08-20 06:10:00         54447  Generator bearing failure  

In [30]:
event_info_filtered = event_info[event_info['asset'] == 0]
event_info_filtered.head()

,asset,event_id,event_label,event_start,event_start_id,event_end,event_end_id,event_description
3,0,73,anomaly,2023-06-10 11:40:00,52745,2023-06-17 11:40:00,53753,Hydraulic group
4,0,0,anomaly,2023-08-06 06:10:00,52436,2023-08-20 06:10:00,54447,Generator bearing failure
5,0,26,anomaly,2023-10-12 10:20:00,52261,2023-10-19 10:20:00,53269,Hydraulic group
14,0,24,normal,2023-04-27 15:00:00,52720,2023-05-11 11:20:00,54714,NaN
18,0,71,normal,2023-01-02 00:00:00,52439,2023-01-16 00:00:00,54455,NaN


,asset,event_id,event_label,event_start,event_start_id,event_end,event_end_id,event_description
0,11,68,anomaly,2023-07-28 13:20:00,52063,2023-08-11 13:10:00,54076,Transformer failure
1,21,22,anomaly,2023-08-12 09:50:00,51888,2023-08-19 10:00:00,52892,Hydraulic group
2,21,72,anomaly,2023-10-10 08:40:00,52497,2023-10-17 08:40:00,53505,Gearbox failure
3,0,73,anomaly,2023-06-10 11:40:00,52745,2023-06-17 11:40:00,53753,Hydraulic group
4,0,0,anomaly,2023-08-06 06:10:00,52436,2023-08-20 06:10:00,54447,Generator bearing failure


In [44]:
# Read data
files = ["0.csv","24.csv","26.csv","71.csv","73.csv"]

data_list = []
for f in files:
    df = pd.read_csv(os.path.join(data_path,f), sep=';')
    df["event_id"] = int(f.replace(".csv",""))
    data_list.append(df)

#Show columns
df.columns

Index(['time_stamp', 'asset_id', 'id', 'train_test', 'status_type_id',
       'sensor_0_avg', 'sensor_1_avg', 'sensor_2_avg', 'wind_speed_3_avg',
       'wind_speed_4_avg', 'wind_speed_3_max', 'wind_speed_3_min',
       'wind_speed_3_std', 'sensor_5_avg', 'sensor_5_max', 'sensor_5_min',
       'sensor_5_std', 'sensor_6_avg', 'sensor_7_avg', 'sensor_8_avg',
       'sensor_9_avg', 'sensor_10_avg', 'sensor_11_avg', 'sensor_12_avg',
       'sensor_13_avg', 'sensor_14_avg', 'sensor_15_avg', 'sensor_16_avg',
       'sensor_17_avg', 'sensor_18_avg', 'sensor_18_max', 'sensor_18_min',
       'sensor_18_std', 'sensor_19_avg', 'sensor_20_avg', 'sensor_21_avg',
       'sensor_22_avg', 'sensor_23_avg', 'sensor_24_avg', 'sensor_25_avg',
       'sensor_26_avg', 'reactive_power_27_avg', 'reactive_power_27_max',
       'reactive_power_27_min', 'reactive_power_27_std',
       'reactive_power_28_avg', 'reactive_power_28_max',
       'reactive_power_28_min', 'reactive_power_28_std', 'power_29_avg',
      

In [33]:
#Concatenate data
raw_data = pd.concat(data_list)
#Convert to datetime and sort by time_stamp
raw_data["time_stamp"] = pd.to_datetime(raw_data["time_stamp"])
raw_data = raw_data.sort_values("time_stamp")
raw_data = raw_data.reset_index(drop=True)
print(raw_data.shape)
raw_data.head()

(272477, 87)


,time_stamp,asset_id,id,train_test,status_type_id,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,...,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg,event_id
0,2022-01-01 00:00:00,0,0,train,0,18.0,178.7,-18.6,4.1,4.4,...,-11991.0,0.0,18831.0,-11991.0,11.1,11.7,10.9,0.1,20.0,71
1,2022-01-01 00:10:00,0,1,train,0,18.0,191.8,-12.2,4.1,4.3,...,-14360.0,0.0,15865.0,-14360.0,11.1,11.3,10.9,0.1,20.0,71
2,2022-01-01 00:20:00,0,2,train,0,18.0,213.8,16.8,4.1,4.4,...,-15524.0,0.0,17244.0,-15524.0,11.1,11.3,10.9,0.1,20.0,71
3,2022-01-01 00:30:00,0,3,train,0,18.0,199.3,-4.6,4.4,4.6,...,-16444.0,0.0,22284.0,-16444.0,11.2,12.4,10.9,0.2,20.0,71
4,2022-01-01 00:40:00,0,4,train,0,18.0,199.9,-4.0,5.5,5.7,...,-16353.0,0.0,49587.0,-16353.0,11.4,12.1,10.9,0.3,20.0,71


In [34]:
# Flag the anomaly events in the raw_data
# status_type_id meanings:
#   0 → Normal Operation  (normal)
#   1 → Derated Operation (anomaly)
#   2 → Idling            (normal)
#   3 → Service           (anomaly)
#   4 → Downtime          (anomaly)
#   5 → Other             (anomaly)

ANOMALY_STATUSES = [1, 3, 4, 5]

raw_data['fault_flag'] = raw_data['status_type_id'].isin(ANOMALY_STATUSES).astype(int)

print(f"Total timesteps        : {len(raw_data):,}")
print(f"Anomaly (fault_flag=1) : {raw_data['fault_flag'].sum():,}")
print(f"Normal  (fault_flag=0) : {(raw_data['fault_flag'] == 0).sum():,}")
raw_data[['time_stamp', 'status_type_id', 'fault_flag']].head(20)

Total timesteps        : 272,477
Anomaly (fault_flag=1) : 35,668
Normal  (fault_flag=0) : 236,809


,time_stamp,status_type_id,fault_flag
0,2022-01-01 00:00:00,0,0
1,2022-01-01 00:10:00,0,0
2,2022-01-01 00:20:00,0,0
3,2022-01-01 00:30:00,0,0
4,2022-01-01 00:40:00,0,0
5,2022-01-01 00:50:00,0,0
6,2022-01-01 01:00:00,0,0
7,2022-01-01 01:10:00,0,0
8,2022-01-01 01:20:00,0,0
9,2022-01-01 01:30:00,0,0


In [35]:
# Also flag anomaly event windows from event_info_filtered (asset_id = 0)
# fault_flag from the previous cell (status_type_id-based) is preserved;
# we only ADD new 1s on top — never reset existing flags.
# We match on event_id and check whether each timestamp falls within [event_start, event_end]

# Ensure event_start / event_end are datetime
event_info_filtered = event_info_filtered.copy()
event_info_filtered['event_start'] = pd.to_datetime(event_info_filtered['event_start'])
event_info_filtered['event_end']   = pd.to_datetime(event_info_filtered['event_end'])

# For every anomaly event, flag the matching timestamps
anomaly_events = event_info_filtered[event_info_filtered['event_label'] == 'anomaly']

for _, row in anomaly_events.iterrows():
    mask = (
        (raw_data['event_id']   == row['event_id']) &
        (raw_data['time_stamp'] >= row['event_start']) &
        (raw_data['time_stamp'] <= row['event_end'])
    )
    raw_data.loc[mask, 'fault_flag'] = 1

print(f"Total timesteps      : {len(raw_data):,}")
print(f"Anomaly  (fault_flag=1): {raw_data['fault_flag'].sum():,}")
print(f"Normal   (fault_flag=0): {(raw_data['fault_flag'] == 0).sum():,}")
raw_data[['time_stamp', 'event_id', 'fault_flag']].head(20)


Total timesteps      : 272,477
Anomaly  (fault_flag=1): 35,668
Normal   (fault_flag=0): 236,809


,time_stamp,event_id,fault_flag
0,2022-01-01 00:00:00,71,0
1,2022-01-01 00:10:00,71,0
2,2022-01-01 00:20:00,71,0
3,2022-01-01 00:30:00,71,0
4,2022-01-01 00:40:00,71,0
5,2022-01-01 00:50:00,71,0
6,2022-01-01 01:00:00,71,0
7,2022-01-01 01:10:00,71,0
8,2022-01-01 01:20:00,71,0
9,2022-01-01 01:30:00,71,0


# Plot Sensor Timeseries with Event Overlay
- Create a function that, given a sensor name (e.g., `sensor_1_avg`) and a time window, plots the sensor values.
- On the same plot, highlight the periods where `fault_flag == 1` (anomaly events) with a shaded background.
- This is essential for exploratory data analysis.

In [46]:
def plot_sensor_timeseries(sensor_name,time_wind):
    pass




    

# EDA

In [ ]:
# Remove anomalous time periods based on event_info

# Convert event times to datetime
event_info["event_start"] = pd.to_datetime(event_info["event_start"])
event_info["event_end"] = pd.to_datetime(event_info["event_end"])

# Select anomaly events only
anomaly_events = event_info[event_info["event_label"] == "anomaly"]

# Create mask to keep normal data
mask = pd.Series(True, index=data.index)

# Remove rows that fall into anomaly event windows
for _, event in anomaly_events.iterrows():
    start = event["event_start"]
    end = event["event_end"]
    mask &= ~((data["time_stamp"] >= start) & (data["time_stamp"] <= end))

# Remove rows with abnormal turbine status
# Keep only status_type == 0 (Normal Operation)
mask &= data["status_type"].isin([0])

# Apply mask
data_filtered = data[mask].copy()

print("Original shape:", data.shape)
print("Shape after removing anomaly periods and abnormal statuses:", data_filtered.shape)

In [ ]:
# Define Metrics

# Train an Unsupervised Anomaly Detector

# Alarm Generation with Persistence Filter

# Plot Results